In [ ]:
# 1. INSTALL LIBRARY (Run this once)
!pip install pypdf

import os
import json
from pypdf import PdfReader
from google.colab import drive

# ==========================================
# 2. MOUNT DRIVE & LOAD DOCUMENTS
# ==========================================

# Mount Google Drive
drive.mount('/content/drive')

def load_pdfs_from_drive(folder_path):
    metadata_store = []

    # Check if folder exists
    if not os.path.exists(folder_path):
        print(f"Error: Folder not found at {folder_path}")
        return []

    print(f"Loading files from: {folder_path}...")

    # Loop through all files in the folder
    files = [f for f in os.listdir(folder_path) if f.endswith('.pdf')]

    for filename in files:
        file_path = os.path.join(folder_path, filename)

        try:
            reader = PdfReader(file_path)
            # Loop through each page in the PDF
            for i, page in enumerate(reader.pages):
                text = page.extract_text()

                # Store in the required format
                doc_entry = {
                    "file_id": f"id_{filename}_{i}", # Generating a simple ID
                    "filename": filename,
                    "page_number": i + 1,
                    "text": text if text else "[NO TEXT FOUND]"
                }
                metadata_store.append(doc_entry)

        except Exception as e:
            print(f"Could not read {filename}: {e}")

    print(f"Successfully loaded {len(metadata_store)} pages from {len(files)} files.")
    return metadata_store

# --- UPDATE THIS PATH TO YOUR ACTUAL DRIVE FOLDER ---
# Example: "/content/drive/MyDrive/Extern_Project/PDF_Data"
folder_path = "/content/drive/MyDrive/Files"

# Load the real data
pdf_metadata_store = load_pdfs_from_drive(folder_path)

# ==========================================
# 3. LLM CLASSIFICATION (Replace with your Key)
# ==========================================

# KNOWN_TYPES from your assignment
KNOWN_TYPES = ["pay_stub", "loan_form", "resume", "contract", "w2", "unknown"]

def get_llm_response(prompt):
    """
    To make this work with the 8 real files, you should use a real LLM here.
    Example using OpenAI (if you have a key):
    """
    # import openai
    # client = openai.OpenAI(api_key="YOUR_KEY_HERE")
    # response = client.chat.completions.create(
    #     model="gpt-3.5-turbo",
    #     messages=[{"role": "user", "content": prompt}],
    #     temperature=0
    # )
    # return response.choices[0].message.content

    # --- SIMULATED RESPONSE (Delete this if using real API) ---
    # This acts as a dummy for testing without a key
    p_lower = prompt.lower()
    if "salary" in p_lower: return "pay_stub"
    if "loan" in p_lower: return "loan_form"
    if "resume" in p_lower: return "resume"
    return "unknown"
    # --------------------------------------------------------

def classify_query_llm(query):
    prompt = f"Classify this query into one of {KNOWN_TYPES}: '{query}'. Return ONLY the category."
    return get_llm_response(prompt).strip().lower()

def classify_doc_type_llm(page_text):
    # Truncate text to avoid token limits
    prompt = f"Classify this document text into one of {KNOWN_TYPES}: '{page_text[:1000]}...'. Return ONLY the category."
    return get_llm_response(prompt).strip().lower()

# ==========================================
# 4. ROUTING LOGIC
# ==========================================

def route_query(user_query):
    # Step A: Predict what document type the user needs
    predicted_type = classify_query_llm(user_query)
    print(f"Query: '{user_query}' -> Looking for: {predicted_type}")

    matched_documents = []

    # Step B: Check our loaded PDFs
    for page in pdf_metadata_store:
        # Classify the page content
        # (In a real app, you'd cache this so you don't re-classify every time)
        page_category = classify_doc_type_llm(page["text"])

        if page_category == predicted_type:
            # Add the doc_type to the object for the final output
            page["doc_type"] = page_category
            matched_documents.append(page)

    # Step C: Fallback (if LLM fails, use keywords)
    if not matched_documents:
        print("No matches by category. Trying keywords...")
        keywords = user_query.lower().split()
        for page in pdf_metadata_store:
            if any(k in page["text"].lower() for k in keywords if len(k) > 4):
                 page["doc_type"] = "keyword_match"
                 matched_documents.append(page)

    return {
        "query": user_query,
        "predicted_doc_type": predicted_type,
        "matched_documents": matched_documents
    }

# ==========================================
# 5. TEST
# ==========================================

# Only run if we actually loaded files
if pdf_metadata_store:
    result = route_query("What is my monthly salary?")
    print(json.dumps(result, indent=4))
else:
    print("Please update 'folder_path' to point to your Drive folder containing the 8 PDFs.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading files from: /content/drive/MyDrive/Files...
Successfully loaded 20 pages from 7 files.
Query: 'What is my monthly salary?' -> Looking for: pay_stub
No matches by category. Trying keywords...
{
    "query": "What is my monthly salary?",
    "predicted_doc_type": "pay_stub",
    "matched_documents": [
        {
            "file_id": "id_functionalsample.pdf_0",
            "filename": "functionalsample.pdf",
            "page_number": 1,
            "text": "Functional Resume Sample \n \nJohn W. Smith  \n2002 Front Range Way Fort Collins, CO 80525  \njwsmith@colostate.edu \n \nCareer Summary \n \nFour years experience in early childhood development with a diverse background in the care of \nspecial needs children and adults.  \n  \nAdult Care Experience \n \n\u2022 Determined work placement for 150 special needs adult clients.  \n\u2022 Maintained clie